In [ ]:
# Validacao do contrato de dados do MVP (temporada 2022).
# Os tipos abaixo sao observados nos objetos retornados pelo FastF1.
from collections import OrderedDict
from pathlib import Path

import fastf1
import pandas as pd

CACHE_DIR = Path("../cache/fastf1")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

MVP_SEASON = 2022
MVP_ROUND = 1
MVP_SESSION_TYPE = "R"

mvp_session = fastf1.get_session(MVP_SEASON, MVP_ROUND, MVP_SESSION_TYPE)
mvp_session.load(weather=True, messages=False)
mvp_laps = mvp_session.laps
mvp_fastest_lap = mvp_laps.pick_fastest()
mvp_telemetry = mvp_fastest_lap.get_telemetry()
mvp_weather = mvp_laps.get_weather_data()
mvp_events = fastf1.get_event_schedule(MVP_SEASON, include_testing=False)

mvp_tables = OrderedDict(
    seasons=pd.DataFrame({"season": [MVP_SEASON]}),
    events=mvp_events,
    sessions=pd.DataFrame(
        [
            {
                "season": MVP_SEASON,
                "round": mvp_session.event.RoundNumber,
                "session_type": MVP_SESSION_TYPE,
                "session_name": mvp_session.name,
                "date": mvp_session.date,
            }
        ]
    ),
    drivers=mvp_session.results,
    laps=mvp_laps,
    telemetry=mvp_telemetry,
    weather=mvp_weather,
)

for table_name, table in mvp_tables.items():
    print(f"[{table_name}] rows={len(table)}")
    print(
        pd.DataFrame(
            {
                "column": table.columns,
                "dtype": [str(table[column].dtype) for column in table.columns],
                "nullable": [bool(table[column].isna().any()) for column in table.columns],
            }
        ).to_string(index=False)
    )
    print()
